<a href="https://colab.research.google.com/github/ShuhaievaPolina/UAV-Visual-Localization-via-Satellite-Imagery/blob/main/uav_visloc_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visual Localization

**Author:** Shuhaieva Polina Oleksiivna

**Estimated full pipeline runtime:** 5 minutes

**Description of the chosen approach:**

This work uses a two-stage approach to visual localization that combines global search over visual embeddings with local keypoint matching. First, the satellite image is split into tiles, and a feature vector is computed for each tile. The same type of vector is computed for the UAV image, after which the most similar satellite regions are identified using cosine similarity. For the top candidates, an additional local matching step is performed to refine the drone's position within the selected tile. This approach was chosen because of the significant difference between UAV images and the satellite map: a drone image can differ from an orthophoto in scale, camera tilt, lighting, season, and perspective distortion. Because of this, matching based only on individual pixels or simple local features is unstable. The embedding model allows comparing the overall content of images — for example, the shape of roads, field boundaries, and the location of buildings, forests, and water bodies. This makes it possible to quickly narrow the search area from the whole map down to a few most likely regions. At the same time, embeddings describe an image globally and do not always allow precise localization within a tile. Therefore, after the initial search, keypoint matching is applied using ALIKED and LightGlue. Based on the found correspondences, a homography is built, through which the center of the UAV image is projected onto the satellite tile. Thus, DINOv2 is responsible for fast and robust global search, while LightGlue handles local coordinate refinement.

# Pipeline diagram

1. Map tiling
2. Computing embeddings for all tiles
3. Building the satellite candidate database
4. Computing the embedding for the UAV image
5. Computing image-to-tile similarity via cosine similarity
6. Selecting the top-5 best tiles
7. Applying ALIKED keypoint detection
8. LightGlue matching
9. Homography estimation with USAC_MAGSAC
10. If a homography is found with enough inliers:
    Project the image center onto the tile using the homography

    If not found:
    Project the image center onto the center of the top-1 tile
11. Computing global pixel coordinates
12. Computing geographic coordinates
13. Computing the Haversine error
14. Visualization and metrics output

# Architecture used

The pretrained model vit_small_patch14_dinov2.lvd142m is used for global search. This is a Vision Transformer from the DINOv2 family, trained with self-supervised learning on a large collection of images. The model's classification head is not used, so the output for each image is a global embedding describing its main visual content.

Before being fed into the model, UAV images and satellite tiles are resized to 518 x 518, converted to tensors, and normalized using ImageNet statistics. The resulting embeddings are additionally L2-normalized. Similarity between the UAV image and each satellite tile is computed using cosine similarity — the higher the value, the more visually similar the images are considered. For local refinement, ALIKED is used to detect up to 1024 keypoints, and LightGlue establishes correspondences between the keypoints of the UAV image and the satellite tile. Based on the found correspondences, a homography is estimated using USAC_MAGSAC. The candidate for the final prediction is the tile with the highest number of geometrically consistent correspondences.

# Tiling parameters

The satellite map is split into square tiles of 1024 x 1024 pixels. The overlap between neighboring tiles is 50%, so the stride equals 512 pixels. Overlap reduces the risk of the target area falling on the boundary between two tiles and lacking sufficient spatial context. A tile size of 1024 pixels is a compromise between the amount of context available for global search and the localization accuracy within a tile.

# Loading libraries

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

!pip install -q torch torchvision opencv-python pandas matplotlib rasterio timm
!pip install -q git+https://github.com/cvg/LightGlue.git

import torch
import rasterio
import cv2
import timm
from torchvision import transforms
from IPython.display import display
from lightglue import LightGlue, ALIKED
from lightglue.utils import load_image, rbd

# Loading the dataset

In [ ]:
if IN_COLAB:
    drive.mount('/content/drive', force_remount=True)

ZIP_PATH = '/content/drive/MyDrive/UAV_VisLoc_dataset.zip'

if os.path.exists(ZIP_PATH):
    !unzip -q -o "{ZIP_PATH}" "*06/*" "*.csv" -d /content/
    print("Extraction complete")
else:
    print("Archive not found, extraction skipped")

data_dir = '/content/06'
drone_dir = os.path.join(data_dir, 'drone')
satmap_path = os.path.join(data_dir, 'satellite06.tif')
coord_path = '/content/satellite_ coordinates_range.csv'  # NOTE: filename contains a space, matches the original dataset archive

if not os.path.exists(satmap_path):
    raise FileNotFoundError(f"File not found: {satmap_path}")

if not os.path.exists(coord_path):
    raise FileNotFoundError(f"File not found: {coord_path}")

# Setting the seed

In [ ]:
def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Coordinate calibration

In [ ]:
class GeoCalibrator:
    def __init__(self, coord_range_path, sat_tif_path, target_map="06"):
        self.coord_range_path = coord_range_path
        self.sat_tif_path = sat_tif_path
        self.target_map = target_map
        self._load_geo_params()

    def _load_geo_params(self):
        with rasterio.open(self.sat_tif_path) as src:
            self.width = src.width
            self.height = src.height

        coords_df = pd.read_csv(self.coord_range_path)
        match = coords_df[coords_df['mapname'].astype(str).str.contains(self.target_map, na=False)]

        if match.empty:
            raise ValueError(f"Could not find coordinates for map: {self.target_map}")

        target_row = match.iloc[0]

        self.lat_max = float(target_row["LT_lat_map"])
        self.lon_min = float(target_row["LT_lon_map"])
        self.lat_min = float(target_row["RB_lat_map"])
        self.lon_max = float(target_row["RB_lon_map"])

    def pixels_to_wgs84(self, x, y):
        lat = self.lat_max - (y / self.height) * (self.lat_max - self.lat_min)
        lon = self.lon_min + (x / self.width) * (self.lon_max - self.lon_min)
        return float(lat), float(lon)

    def wgs84_to_pixels(self, lat, lon):
        y = (self.lat_max - lat) / (self.lat_max - self.lat_min) * self.height
        x = (lon - self.lon_min) / (self.lon_max - self.lon_min) * self.width
        return float(x), float(y)

# Haversine error (HME) calculation

In [ ]:
def hme(lat, lon, pred_lat, pred_lon):
    R = 6371000.0
    phi1, phi2 = math.radians(lat), math.radians(pred_lat)
    delta_phi = math.radians(pred_lat - lat)
    delta_lambda = math.radians(pred_lon - lon)
    a = (
        math.sin(delta_phi / 2.0) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2.0) ** 2
    )
    return R * (2 * math.atan2(math.sqrt(a), math.sqrt(1 - a)))

# Map tiling

In [ ]:
class SatelliteTileDatabase:
    def __init__(self, sat_path, tile_size=1024, overlap=0.5):
        self.sat_path = sat_path
        self.tile_size = tile_size
        self.stride = int(tile_size * (1 - overlap))
        self.tiles_meta = []
        self.embeddings = None

        self.coarse_model = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0).to(device)
        self.coarse_model.eval()

        self.transform = transforms.Compose([
            transforms.Resize((518, 518)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def generate_tiles_and_index(self):
        print("\nTiling the map")
        with rasterio.open(self.sat_path) as src:
            sat_img = src.read().transpose(1, 2, 0)

        h, w, _ = sat_img.shape
        tensors = []

        for y in range(0, h - self.tile_size + 1, self.stride):
            for x in range(0, w - self.tile_size + 1, self.stride):
                tile = sat_img[y:y+self.tile_size, x:x+self.tile_size]
                self.tiles_meta.append({'x': x, 'y': y, 'image': tile})

                pil_tile = Image.fromarray(tile)
                tensors.append(self.transform(pil_tile))

        print(f"Created {len(self.tiles_meta)} tiles")

        batch_size = 32
        all_embs = []
        with torch.no_grad():
            for i in range(0, len(tensors), batch_size):
                batch = torch.stack(tensors[i:i+batch_size]).to(device)
                emb = self.coarse_model(batch)
                emb = torch.nn.functional.normalize(emb, p=2, dim=1)
                all_embs.append(emb.cpu())

        self.embeddings = torch.cat(all_embs, dim=0).to(device)

    def query_top_k(self, drone_img_path, k=5):
        pil_img = Image.open(drone_img_path).convert('RGB')
        tensor = self.transform(pil_img).unsqueeze(0).to(device)

        with torch.no_grad():
            query_emb = self.coarse_model(tensor)
            query_emb = torch.nn.functional.normalize(query_emb, p=2, dim=1)

        sims = torch.mm(query_emb, self.embeddings.t()).squeeze(0)
        topk = torch.topk(sims, k=k)

        topk_idx = topk.indices.cpu().numpy()
        topk_scores = topk.values.cpu().numpy()

        results = []
        for idx, score in zip(topk_idx, topk_scores):
            tile_meta = self.tiles_meta[idx].copy()
            tile_meta['score'] = float(score)
            results.append(tile_meta)

        return results

# Keypoint detection and matching

In [ ]:
class EdgeLightGlueMatcher:
    def __init__(self, max_num_keypoints=1024, sim_threshold=0.82, early_stop_inliers=25):
        self.extractor = ALIKED(max_num_keypoints=max_num_keypoints).eval().to(device)
        self.matcher = LightGlue(features="aliked", depth_confidence=0.9, width_confidence=0.95).eval().to(device)
        self.sim_threshold = sim_threshold
        self.early_stop_inliers = early_stop_inliers

    def match_and_find_homography(self, drone_img_path, candidate_tiles, max_candidates=3):
        candidates = candidate_tiles[:max_candidates]

        if candidates and "score" in candidates[0] and candidates[0]["score"] >= self.sim_threshold:
            candidates = [candidates[0]]

        image0 = load_image(drone_img_path).to(device)
        with torch.no_grad():
            feats0 = self.extractor.extract(image0)

        best_inliers = 0
        best_homography = None
        best_tile = None


        for tile_meta in candidates:
            tile_tensor = torch.from_numpy(tile_meta["image"]).permute(2, 0, 1).float() / 255.0
            image1 = tile_tensor.to(device)

            with torch.no_grad():
                feats1 = self.extractor.extract(image1)
                matches01 = self.matcher({"image0": feats0, "image1": feats1})

            feats0_clean, feats1_clean, matches01_clean = [rbd(x) for x in [feats0, feats1, matches01]]
            matches = matches01_clean["matches"]
            points0 = feats0_clean["keypoints"][matches[..., 0]].cpu().numpy()
            points1 = feats1_clean["keypoints"][matches[..., 1]].cpu().numpy()

            if len(points0) < 4:
                continue

            H, mask = cv2.findHomography(points0, points1, cv2.USAC_MAGSAC, 5.0)

            if H is not None:
                inliers = mask.sum()
                if inliers > best_inliers:
                    best_inliers = inliers
                    best_homography = H
                    best_tile = tile_meta

                if best_inliers >= self.early_stop_inliers:
                    break

        return best_tile, best_homography, best_inliers

# Sample image check

In [ ]:
sample_images = sorted(os.listdir(drone_dir))[:3]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, img_name in enumerate(sample_images):
    img_p = os.path.join(drone_dir, img_name)
    img_bgr = cv2.imread(img_p)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img_rgb)
    axes[i].set_title(f"Test image: {img_name}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

# Main pipeline

In [ ]:
def run_evaluation_pipeline(data_dir, satmap_path, coord_path, drone_dir):
    gt_csv_path = os.path.join(data_dir, "06.csv")

    calibrator = GeoCalibrator(coord_path, satmap_path)
    sat_db = SatelliteTileDatabase(satmap_path, tile_size=1024, overlap=0.5)
    sat_db.generate_tiles_and_index()

    matcher = EdgeLightGlueMatcher(max_num_keypoints=1024)
    gt_df = pd.read_csv(gt_csv_path)

    drone_images = sorted([os.path.join(drone_dir, f) for f in os.listdir(drone_dir)])

    results = []
    top1_hits = 0
    top5_hits = 0

    for img_path in tqdm(drone_images, desc="Localization"):
        image_name = os.path.basename(img_path)

        gt_row = gt_df[gt_df['filename'] == image_name]
        if gt_row.empty:
            continue

        gt_lat = float(gt_row['lat'].values[0])
        gt_lon = float(gt_row['lon'].values[0])

        top_tiles = sat_db.query_top_k(img_path, k=5)
        top1_similarity = top_tiles[0]['score']

        gt_px, gt_py = calibrator.wgs84_to_pixels(gt_lat, gt_lon)

        def is_in_tile(tile):
            return (tile['x'] <= gt_px <= tile['x'] + sat_db.tile_size) and \
                   (tile['y'] <= gt_py <= tile['y'] + sat_db.tile_size)

        if is_in_tile(top_tiles[0]):
            top1_hits += 1

        if any(is_in_tile(t) for t in top_tiles[:5]):
            top5_hits += 1

        best_tile, H, inliers = matcher.match_and_find_homography(img_path, top_tiles)

        MIN_INLIERS = 6

        if best_tile is not None and H is not None and inliers >= MIN_INLIERS:
            drone_bgr = cv2.imread(img_path)
            img_h, img_w = drone_bgr.shape[:2] if drone_bgr is not None else (518, 518)

            center_drone = np.array([[[img_w / 2.0, img_h / 2.0]]], dtype=np.float32)
            center_tile = cv2.perspectiveTransform(center_drone, H)[0][0]

            global_x = best_tile['x'] + center_tile[0]
            global_y = best_tile['y'] + center_tile[1]
        else:
            best_tile = top_tiles[0]
            global_x = best_tile['x'] + (sat_db.tile_size / 2.0)
            global_y = best_tile['y'] + (sat_db.tile_size / 2.0)
            inliers = 0

        pred_lat, pred_lon = calibrator.pixels_to_wgs84(global_x, global_y)
        error_m = hme(gt_lat, gt_lon, pred_lat, pred_lon)

        results.append({
            'filename': image_name,
            'pred_lat': pred_lat,
            'pred_lon': pred_lon,
            'gt_lat': gt_lat,
            'gt_lon': gt_lon,
            'error_m': error_m,
            'top1_similarity': top1_similarity,
            'inliers': inliers
        })

    df_res = pd.DataFrame(results)
    total_eval = len(df_res)
    recall_1 = (top1_hits / total_eval * 100) if total_eval > 0 else 0
    recall_5 = (top5_hits / total_eval * 100) if total_eval > 0 else 0

    return df_res, calibrator, satmap_path, recall_1, recall_5

# Metrics output and visualization

In [ ]:
def print_metrics_and_plot(df, calibrator, sat_path, recall_1, recall_5):
    mhe = df['error_m'].mean()
    median_err = df['error_m'].median()
    p90_err = np.percentile(df['error_m'], 90)

    hit_50m = (df['error_m'] <= 50).mean() * 100
    hit_100m = (df['error_m'] <= 100).mean() * 100
    hit_500m = (df['error_m'] <= 500).mean() * 100


    print("\nFINAL METRICS (UAV-VisLoc Subset 06)\n")
    print(f"Number of test images:           {len(df)}")
    print(f"Mean Haversine Error:            {mhe:.2f} m")
    print(f"Median Error:                    {median_err:.2f} m")
    print(f"90th Percentile (P90):           {p90_err:.2f} m")
    print(f"Hit Rate (<= 50 m):              {hit_50m:.2f} %")
    print(f"Hit Rate (<= 100 m):             {hit_100m:.2f} %")
    print(f"Hit Rate (<= 500 m):             {hit_500m:.2f} %")
    print(f"Top-1 Hit Rate:                  {recall_1:.2f} %")
    print(f"Top-5 Hit Rate:                  {recall_5:.2f} %")

    export_cols = ['filename', 'pred_lat', 'pred_lon', 'gt_lat', 'gt_lon', 'error_m', 'top1_similarity']
    df_export = df[export_cols]
    df_export.to_csv("predictions_06.csv", index=False)
    print("\nResults saved to file: predictions_06.csv")

    print("\nMap visualization\n")
    with rasterio.open(sat_path) as src:
        sat_img = src.read(1)

    plt.figure(figsize=(12, 10))
    plt.imshow(sat_img, cmap='gray')

    gt_pxs = [calibrator.wgs84_to_pixels(lat, lon) for lat, lon in zip(df['gt_lat'], df['gt_lon'])]
    pred_pxs = [calibrator.wgs84_to_pixels(lat, lon) for lat, lon in zip(df['pred_lat'], df['pred_lon'])]

    gt_x, gt_y = zip(*gt_pxs)
    pred_x, pred_y = zip(*pred_pxs)

    for gx, gy, px, py in zip(gt_x, gt_y, pred_x, pred_y):
        plt.plot([gx, px], [gy, py], color='red', alpha=0.5, linewidth=1)

    plt.scatter(gt_x, gt_y, c='lime', label='Ground Truth', s=25, zorder=3)
    plt.scatter(pred_x, pred_y, c='blue', label='Predicted', s=20, zorder=4)

    plt.title("Localization accuracy visualization (Green = GT, Blue = Pred, Red = Error)")
    plt.legend()
    plt.axis('off')

    h, w = sat_img.shape
    plt.xlim(0, w)
    plt.ylim(h, 0)

    plt.tight_layout()
    plt.savefig("localization_map_06.png", dpi=200)
    plt.show()


def visualize_top_results(df_subset, title_prefix, drone_dir, sat_path, calibrator, crop_size=1024, show_full_map=False):
    with rasterio.open(sat_path) as src:
        sat_img = src.read().transpose(1, 2, 0)

    sat_h, sat_w, _ = sat_img.shape

    for idx, row in df_subset.iterrows():
        img_name = row['filename']
        drone_path = os.path.join(drone_dir, img_name)

        drone_bgr = cv2.imread(drone_path)
        if drone_bgr is not None:
            drone_rgb = cv2.cvtColor(drone_bgr, cv2.COLOR_BGR2RGB)
        else:
            drone_rgb = None

        gt_px, gt_py = calibrator.wgs84_to_pixels(row['gt_lat'], row['gt_lon'])
        pred_px, pred_py = calibrator.wgs84_to_pixels(row['pred_lat'], row['pred_lon'])

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        if drone_rgb is not None:
            axes[0].imshow(drone_rgb)
            axes[0].set_title(f"Image: {img_name}\nSim: {row['top1_similarity']:.3f}   Inliers: {row['inliers']}")
        else:
            axes[0].text(0.5, 0.5, f"File not found:\n{img_name}", ha='center', va='center')
        axes[0].axis('off')

        if show_full_map:
            axes[1].imshow(sat_img)
            axes[1].scatter([gt_px], [gt_py], c='lime', s=120, marker='o', edgecolors='black', label='GT')
            axes[1].scatter([pred_px], [pred_py], c='red', s=120, marker='X', edgecolors='black', label='Pred')
            axes[1].plot([gt_px, pred_px], [gt_py, pred_py], 'yellow', linestyle='--', linewidth=2, label=f"MHE: {row['error_m']:.1f} m")

            axes[1].set_xlim(0, sat_w)
            axes[1].set_ylim(sat_h, 0)
            axes[1].set_title(f"{title_prefix}  MHE = {row['error_m']:.2f} m")
        else:
            half_crop = crop_size // 2
            x1 = max(0, min(sat_w, int(gt_px - half_crop)))
            y1 = max(0, min(sat_h, int(gt_py - half_crop)))
            x2 = max(0, min(sat_w, int(gt_px + half_crop)))
            y2 = max(0, min(sat_h, int(gt_py + half_crop)))

            sat_patch = sat_img[y1:y2, x1:x2]

            gt_patch_x = gt_px - x1
            gt_patch_y = gt_py - y1
            pred_patch_x = pred_px - x1
            pred_patch_y = pred_py - y1

            if sat_patch.size > 0 and sat_patch.shape[0] > 0 and sat_patch.shape[1] > 0:
                axes[1].imshow(sat_patch)
                axes[1].scatter([gt_patch_x], [gt_patch_y], c='lime', s=120, marker='o', edgecolors='black', label='GT')
                axes[1].scatter([pred_patch_x], [pred_patch_y], c='red', s=120, marker='X', edgecolors='black', label='Pred')
                axes[1].plot([gt_patch_x, pred_patch_x], [gt_patch_y, pred_patch_y], 'yellow', linestyle='--', linewidth=2, label=f"MHE: {row['error_m']:.1f} m")
                axes[1].set_title(f"{title_prefix}   MHE = {row['error_m']:.2f} m")
            else:
                axes[1].text(0.5, 0.5, "GT coordinates are outside\nthe satellite map", ha='center', va='center', color='red', fontsize=12)
                axes[1].set_title(f"{title_prefix}   MHE = {row['error_m']:.2f} m")

        axes[1].legend(loc='upper right')
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()

# Running the pipeline

In [ ]:
results_df, calibrator, satmap_path, rec1, rec5 = run_evaluation_pipeline(
    data_dir=data_dir, satmap_path=satmap_path,
    coord_path=coord_path, drone_dir=drone_dir
)
print_metrics_and_plot(results_df, calibrator, satmap_path, rec1, rec5)

best_5 = results_df.nsmallest(5, 'error_m')
worst_5 = results_df.nlargest(5, 'error_m')

print("\nTop-5 best results:")
display(best_5[['filename', 'error_m', 'top1_similarity', 'inliers', 'pred_lat', 'pred_lon', 'gt_lat', 'gt_lon']])
visualize_top_results(best_5, "Best result", drone_dir, satmap_path, calibrator)

print("\nTop-5 worst results:")
display(worst_5[['filename', 'error_m', 'top1_similarity', 'inliers', 'pred_lat', 'pred_lon', 'gt_lat', 'gt_lon']])
visualize_top_results(worst_5, "Worst result", drone_dir, satmap_path, calibrator, show_full_map=True)

# Results analysis

The best results share a common pattern: embeddings correctly identify the right map region, while ALIKED and LightGlue refine the coordinates within the tile. The most accurate predictions were obtained for scenes with distinct landmarks — roads, buildings, and field or forest boundaries. A high number of inliers usually corresponds to a stable homography and a small error. At the same time, not only the number of inliers matters, but also their quality and distribution across the image. Cosine similarity has no direct relationship with the final error: what matters most is that the correct tile makes it into the top-k, after which local matching provides accuracy down to a few meters.

For the worst results, local matching failed completely, since in all five cases the number of inliers was zero. Because of this, the coordinates were determined by the center of the top-1 tile, and the final accuracy depended entirely on the correctness of the DINOv2 global search. These images are dominated by forests, water bodies, fields, and terraced areas without sufficiently unique landmarks. Such structures repeat across different parts of the map, so the model finds a visually similar but geographically incorrect region. High similarity values also do not guarantee a correct match: even a relatively high similarity score can correspond to a different area with a similar type of terrain. Thus, the main causes of large errors are the absence of stable keypoints and the use of the center of an incorrectly selected tile as the coordinate prediction.

# Conclusions

This work implements a combined approach to UAV image localization, in which a global search for the corresponding map region is performed first using embeddings, and then the position is refined using local keypoints. The best results were obtained for scenes with clearly defined objects (roads, buildings, distinct field boundaries). The largest errors occurred in homogeneous natural areas, where local matching failed to find reliable correspondences and the prediction was determined by the center of an incorrectly selected tile.

The approach could be improved with multi-scale tiling and by checking a larger number of keypoint candidates. Further accuracy gains are also possible by fine-tuning the embedding model on datasets characteristic of this specific task.